# How to use the Compatibility API

The `/compatibility` endpoint helps you understand how to call TiTiler-CMR for a collection. For hierarchical HDF5 and NetCDF datasets it can now also tell you when a `group` parameter is needed, suggest candidate groups, and return links that already include the sampled group when one was used to inspect the file.

This notebook demonstrates how to interpret that response for the [NISAR Beta Geocoded Polarimetric Covariance (GCOV)](https://www.earthdata.nasa.gov/data/catalog/asf-nisar-l2-gcov-beta-v1-1) collection, which is a known example of a hierarchical HDF5 dataset that needs a `group` parameter for xarray requests.

## Setup

In [ ]:
import json
import os

import earthaccess
import httpx2 as httpx

titiler_endpoint = os.getenv(
    "TITILER_CMR_ENDPOINT", "https://v4jec6i5c0.execute-api.us-west-2.amazonaws.com"
)

## Identify the dataset

You can find the collection with `earthaccess.search_datasets`.

In [ ]:
datasets = earthaccess.search_datasets(concept_id="C3622214170-ASF")
ds = datasets[0]

collection_concept_id = ds["meta"]["concept-id"]
print("CollectionConcept-Id:", collection_concept_id)
print("Short name:", ds["umm"]["ShortName"])
print("Version:", ds["umm"]["Version"])

## Explore the collection using `/compatibility`

In [ ]:
compatibility_response = httpx.get(
    f"{titiler_endpoint}/compatibility",
    params={"collection_concept_id": collection_concept_id},
    timeout=None,
).json()

print(json.dumps(compatibility_response, indent=2))

## Interpret the response

For hierarchical datasets, the most important fields are:

- `requires_group`: the root dataset was not enough, so xarray requests should include `group=...`
- `group_hints`: candidate group paths to try
- `default_group`: a recommended group when there is a clear single choice
- `sampled_group`: the group that was actually used to collect the returned metadata

If `sampled_group` is present, the compatibility links should already include it.

In [ ]:
group_hints = compatibility_response.get("group_hints", [])
sampled_group = compatibility_response.get("sampled_group")
default_group = compatibility_response.get("default_group")
variables = list((compatibility_response.get("variables") or {}).keys())
tilejson_link = next(
    link["href"]
    for link in compatibility_response.get("links", [])
    if link["rel"] == "tilejson"
)

print("requires_group:", compatibility_response.get("requires_group"))
print("group_hints:")
for hint in group_hints:
    print(" -", hint)
print("default_group:", default_group)
print("sampled_group:", sampled_group)
print("variables (first 10):", variables[:10])
print("tilejson link:", tilejson_link)

For a grouped dataset like NISAR GCOV, the response should tell you exactly how to proceed:

1. If `requires_group` is `true`, use a group from `group_hints`.
2. Prefer `default_group` when it is present.
3. If `default_group` is `null`, fall back to `sampled_group` or choose from `group_hints`.
4. Pick a variable name from `variables`.
5. Reuse the returned compatibility links, since they should already include the sampled group when one was needed.

That turns the compatibility response into a practical recipe for the next xarray request instead of leaving you to guess the right group path.